In [15]:
from collections import Counter, defaultdict
from datetime import datetime, timedelta
import re

# Load data
with open("hostel_bois.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

parsed_messages = []
stop_words = {
    "is",
    "the",
    "a",
    "and",
    "to",
    "of",
    "in",
    "on",
    "for",
    "with",
    "it",
    "that",
    "this",
    "was",
    "are",
    "be",
    "my",
    "me",
    "at",
    "he",
    "she",
    "they",
    "but",
    "not",
    "or",
    "as",
    "an",
    "if",
    "so",
    "do",
    "can",
    "just",
    "what",
    "all",
    "out",
    "up",
    "have",
    "like",
    "you",
    "i",
    "your",
    "we",
    "no",
    "yes",
    "from",
    "their",
    "them",
    "who",
    "where",
    "how",
    "why",
    "get",
    "one",
    "go",
    "then",
    "more",
    "time",
}

# 1. Parse & filter messages
for line in lines:
    line = line.strip()
    if not line:
        continue
    parts = line.split(" - ", 1)
    if len(parts) < 2:
        continue
    timestamp = parts[0]
    sender_and_message = parts[1]
    sender_parts = sender_and_message.split(": ", 1)
    if len(sender_parts) < 2:
        continue
    sender = sender_parts[0]
    message_text = sender_parts[1]
    if (
        message_text == "<Media omitted>"
        or message_text == "This message was deleted"
    ):
        continue
    parsed_messages.append(
        {"timestamp": timestamp, "sender": sender, "message": message_text}
    )

total_messages = len(parsed_messages)

# 2. Group Overview Data
participants = sorted(list(set(msg["sender"] for msg in parsed_messages)))
num_participants = len(participants)
message_counts = defaultdict(int)
for msg in parsed_messages:
    message_counts[msg["sender"]] += 1
sorted_participants = sorted(
    message_counts.items(), key=lambda x: x[1], reverse=True
)

dates = [
    datetime.strptime(msg["timestamp"].split(",")[0], "%d/%m/%y")
    for msg in parsed_messages
]
start_date = min(dates).strftime("%d %B %Y")
end_date = max(dates).strftime("%d %B %Y")

# 3. Activity Heatmap Data
participant_to_idx = {p: i for i, p in enumerate(participants)}
heatmap_matrix = np.zeros((len(participants), 24), dtype=int)
for msg in parsed_messages:
    sender = msg["sender"]
    hour = int(msg["timestamp"].split(",")[1].split(":")[0].strip())
    heatmap_matrix[participant_to_idx[sender], hour] += 1

# 4. Top Words Data
all_words = []
for msg in parsed_messages:
    words = re.findall(r"\w+", msg["message"].lower())
    for word in words:
        if word not in stop_words and len(word) > 1:
            all_words.append(word)
word_counts = Counter(all_words)
top_words = word_counts.most_common(5)

# 5. Print the Final Report
print("========================================")
print('GROUPDNA REPORT - "Hostel Bois 4ever"')
print(f"60 days • {total_messages} messages • {num_participants} members")
print("========================================\n")
print("MESSAGES PER PERSON")
for sender, count in sorted_participants:
    percentage = (count / total_messages) * 100
    print(f"{sender}: {count} ({percentage:.1f}%)")
print("\nACTIVITY HEATMAP (messages by hour)")
hours_header = "   " + "".join([f"{h:02d} " for h in range(24)])
print(hours_header)
for p_idx, participant in enumerate(participants):
    row_str = f"{participant[:4]}: "
    for hour in range(24):
        msg_count = heatmap_matrix[p_idx, hour]
        if msg_count == 0:
            char = ". "
        elif msg_count <= 5:
            char = "- "
        elif msg_count <= 15:
            char = "= "
        else:
            char = "# "
        row_str += char
    print(row_str)
print("\nTHIS GROUP'S FAVOURITE WORDS")
for word, count in top_words:
    print(f"{word}: {count}")
print("\n========================================")


GROUPDNA REPORT - "Hostel Bois 4ever"
60 days • 3127 messages • 6 members

MESSAGES PER PERSON
Rahul: 940 (30.1%)
Priya: 712 (22.8%)
Neha: 624 (20.0%)
Aman: 484 (15.5%)
Karan: 345 (11.0%)
Vikas: 22 (0.7%)

ACTIVITY HEATMAP (messages by hour)
   00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Aman: # # # # # . . . . . . . . . = = # - # = = = . # 
Kara: . . . . . . . - = # # # # # # # # # # # # = = = 
Neha: . . . . . # - = # # # # # # # = # # # # # # # # 
Priy: . . . . . . = # # # # # # # # # # # # # # # # = 
Rahu: - = # # # = # # # # # = # # # # # # # # # # # # 
Vika: . . . . . . . - - - - . - - . - - - - - - - - - 

THIS GROUP'S FAVOURITE WORDS
guys: 318
today: 292
about: 274
hai: 268
am: 260

